## Intro to Convolutional Neural Networks (CNNs)

This notebook has been corrected to remove invalid markdown-in-code cells and fix the CNN output-shape example.

In the previous notebooks, we used Multilayer Perceptrons (MLPs) for image classification. These models required us to flatten images into long 1D vectors before feeding them to the network. That worked for simple tasks, but it discarded the image's spatial structure.

CNNs were designed specifically for structured grid-like data such as images. They preserve locality, learn filters that detect edges, textures, and shapes, and share weights across the image so that the same pattern can be detected anywhere.


### Why do we need CNNs?

MLPs flatten images into 1D vectors and lose spatial structure. CNNs preserve locality and learn filters that detect edges, textures, and shapes while sharing weights across the image.

This is the key reason CNNs work so well for vision tasks: they model the fact that neighboring pixels are related and patterns can repeat across an image.

A convolutional layer applies a learned filter over local regions, creating feature maps that highlight important patterns such as vertical edges, corners, and textures.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch

# Compute CIFAR-10 normalization stats
train_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
loader = DataLoader(train_full, batch_size=50000, shuffle=False)
images, _ = next(iter(loader))
mean = images.mean(dim=[0, 2, 3])
std = images.std(dim=[0, 2, 3])
print('Mean:', mean)
print('Std:', std)

# Build train/val/test loaders
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_val_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

class_names = train_val_dataset.classes
train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print('Class names:', class_names)
print('Train batches:', len(train_loader))
print('Validation batches:', len(val_loader))
print('Test batches:', len(test_loader))

for images, labels in train_loader:
    print('Batch shape:', images.shape)
    print('Label shape:', labels.shape)
    break


### LeNet architecture for CIFAR-10

A simple adapted LeNet: Conv -> ReLU -> MaxPool -> Conv -> ReLU -> MaxPool -> Flatten -> FC -> ReLU -> FC -> ReLU -> Output.


In [ ]:
import torch.nn as nn

class LeNet_CIFAR10(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(400, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


In [ ]:
# Correct CNN feature-map shape check
sample_images, _ = next(iter(train_loader))
sample_images = sample_images.to(device)

lenet = LeNet_CIFAR10().to(device)
conv_out = lenet.conv_layers(sample_images)
print('Conv output shape:', conv_out.shape)


### Train and evaluate

The example below is a minimal training loop demonstrating the correct use of the CNN model.


In [ ]:
import torch.optim as optim

model = LeNet_CIFAR10().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    print(f'Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')

model.eval()
with torch.no_grad():
    val_correct = 0
    val_total = 0
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        val_correct += (predicted == labels).sum().item()
        val_total += labels.size(0)
    print(f'Validation accuracy: {val_correct / val_total:.4f}')


### Summary

These fixes address the main problems in the notebook: invalid markdown-in-code cells, the incorrect conv-output assignment, and broken notebook structure.
